# 📅 Dia 9 — Datetime e Manipulação de Datas com Pandas

**Fase 1 — Python Basics** · Data: 2026-10-22 · Autor: Gutemberg Rapôso

Datas estão em toda parte: registros de aula, prazos de entrega, carimbos de tempo em plataformas de ensino, logs de uso de aplicativos de idiomas. Hoje o foco é aprender a **representar, converter, calcular e agregar** informações temporais em Python e pandas.

### 🎯 Objetivos de aprendizagem
1. Usar o módulo `datetime`: `date`, `time`, `datetime` e `timedelta`.
2. Converter texto ↔ data com `strptime` / `strftime` (e formato **ISO 8601**).
3. Trabalhar com `pd.to_datetime()`, `DatetimeIndex` e o acessor `.dt`.
4. Fazer **resampling** e agrupamentos por período.
5. Realizar aritmética de datas e lidar com **fusos horários** (`zoneinfo`).
6. Aplicar tudo em uma análise de **progresso de aprendizagem de inglês** ao longo de um ano letivo real (calendário 2026).

## 1. Configuração do ambiente

Bibliotecas usadas: `datetime` e `zoneinfo` (biblioteca padrão), `pandas`, `numpy`, `matplotlib` e `seaborn`.

> 💡 `zoneinfo` faz parte da biblioteca padrão desde o Python 3.9 e substitui, na maioria dos casos, o pacote externo `pytz`.

In [1]:
from datetime import date, time, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 2.2.3 | numpy 1.26.4 | seaborn 0.13.2


## 2. O módulo `datetime`: os quatro tipos fundamentais

| Tipo | Representa | Exemplo |
|---|---|---|
| `date` | uma data de calendário (ano, mês, dia) | início do semestre |
| `time` | um horário do dia, sem data | horário da aula |
| `datetime` | data **e** horário | envio de uma redação |
| `timedelta` | uma **duração** (diferença entre instantes) | tempo até a prova |

In [2]:
semester_start = date(2026, 2, 9)            # first school day (Monday)
class_time = time(19, 30)                    # evening English class
essay_submitted = datetime(2026, 3, 14, 22, 47, 5)
study_session = timedelta(hours=1, minutes=15)

print("Semester start:", semester_start, "| weekday:", semester_start.strftime("%A"))
print("Class time:", class_time)
print("Essay submitted:", essay_submitted)
print("Study session length:", study_session, "=", study_session.total_seconds() / 60, "minutes")

Semester start: 2026-02-09 | weekday: Monday
Class time: 19:30:00
Essay submitted: 2026-03-14 22:47:05
Study session length: 1:15:00 = 75.0 minutes


### 2.1 Combinando e decompondo objetos

`datetime.combine()` junta uma `date` e um `time`. O caminho inverso usa `.date()` e `.time()`. Atributos como `.year`, `.month`, `.isoweekday()` permitem extrair componentes.

> 📝 `weekday()` retorna 0 = segunda … 6 = domingo; `isoweekday()` retorna 1 = segunda … 7 = domingo (padrão ISO).

In [3]:
first_class = datetime.combine(semester_start, class_time)
print("First class:", first_class)
print("Year:", first_class.year, "| Month:", first_class.month, "| Day:", first_class.day)
print("weekday():", first_class.weekday(), "| isoweekday():", first_class.isoweekday())
print("ISO calendar (year, week, weekday):", tuple(first_class.isocalendar()))
print("Back to parts:", first_class.date(), first_class.time())

First class: 2026-02-09 19:30:00
Year: 2026 | Month: 2 | Day: 9
weekday(): 0 | isoweekday(): 1
ISO calendar (year, week, weekday): (2026, 7, 1)
Back to parts: 2026-02-09 19:30:00


### 2.2 `timedelta`: aritmética de datas

Somar ou subtrair `timedelta` de uma data gera outra data. Subtrair duas datas gera um `timedelta`. Isso resolve perguntas como *"quantos dias faltam para a prova?"*.

In [4]:
final_exam = date(2026, 6, 26)
days_left = final_exam - semester_start
print("Days from semester start to final exam:", days_left.days)
print("Weeks (float):", round(days_left.days / 7, 1))

# Deadline 10 days after a given assignment date
assignment = date(2026, 4, 1)
print("Deadline:", assignment + timedelta(days=10))

# Total weekly study time for 3 sessions
weekly = study_session * 3
print("Weekly study time:", weekly, "->", weekly.total_seconds() / 3600, "hours")

Days from semester start to final exam: 137
Weeks (float): 19.6
Deadline: 2026-04-11
Weekly study time: 3:45:00 -> 3.75 hours


## 3. Parsing e formatação: `strptime` e `strftime`

- **`strptime`** (*string parse time*): texto → `datetime`, informando o formato.
- **`strftime`** (*string format time*): `datetime` → texto no formato desejado.

Códigos mais comuns: `%d` dia, `%m` mês, `%Y` ano com 4 dígitos, `%H` hora (24h), `%M` minuto, `%S` segundo, `%A` nome do dia, `%B` nome do mês.

> ⚠️ No Brasil escrevemos `22/10/2026` (dia/mês/ano); nos EUA, `10/22/2026` (mês/dia/ano). Ambiguidade de datas é um problema real em dados educacionais internacionais!

In [5]:
br_text = "22/10/2026 19:30"
us_text = "10/22/2026 7:30 PM"

br_dt = datetime.strptime(br_text, "%d/%m/%Y %H:%M")
us_dt = datetime.strptime(us_text, "%m/%d/%Y %I:%M %p")
print("Parsed BR:", br_dt)
print("Parsed US:", us_dt)
print("Same instant?", br_dt == us_dt)

print("strftime BR:", br_dt.strftime("%d/%m/%Y às %Hh%M"))
print("strftime US:", br_dt.strftime("%B %d, %Y at %I:%M %p"))

Parsed BR: 2026-10-22 19:30:00
Parsed US: 2026-10-22 19:30:00
Same instant? True
strftime BR: 22/10/2026 às 19h30
strftime US: October 22, 2026 at 07:30 PM


### 3.1 Ambiguidade: o mesmo texto, duas datas

A string `"03/04/2026"` pode ser 3 de abril (BR) ou 4 de março (EUA). Por isso, **sempre documente o formato** de origem dos dados.

In [6]:
ambiguous = "03/04/2026"
as_br = datetime.strptime(ambiguous, "%d/%m/%Y").date()
as_us = datetime.strptime(ambiguous, "%m/%d/%Y").date()
print(f"{ambiguous!r} as BR -> {as_br} | as US -> {as_us} | difference: {abs((as_br - as_us).days)} days")

'03/04/2026' as BR -> 2026-04-03 | as US -> 2026-03-04 | difference: 30 days


### 3.2 ISO 8601: o formato que elimina ambiguidade

O padrão **ISO 8601** (`AAAA-MM-DDTHH:MM:SS±HH:MM`) é ordenável como texto, não ambíguo e aceito por praticamente todas as ferramentas. É a **boa prática** para armazenar e trocar datas.

In [7]:
print("isoformat():", essay_submitted.isoformat())
print("fromisoformat():", datetime.fromisoformat("2026-10-22T19:30:00"))
print("date.fromisoformat():", date.fromisoformat("2026-12-18"))

# ISO strings sort chronologically even as plain text
iso_dates = ["2026-11-03", "2026-02-09", "2026-07-20"]
print("Sorted ISO strings:", sorted(iso_dates))

isoformat(): 2026-03-14T22:47:05
fromisoformat(): 2026-10-22 19:30:00
date.fromisoformat(): 2026-12-18
Sorted ISO strings: ['2026-02-09', '2026-07-20', '2026-11-03']


## 4. Fusos horários com `zoneinfo`

Um `datetime` sem fuso é **naive** (ingênuo); com fuso é **aware** (consciente). Para aulas online entre Brasil e outros países, misturar horários naive é fonte clássica de erros.

> 🌎 Tocantins usa `America/Araguaina` (UTC−3, sem horário de verão). Brasília usa `America/Sao_Paulo`.

In [8]:
naive = datetime(2026, 10, 22, 19, 30)
palmas = naive.replace(tzinfo=ZoneInfo("America/Araguaina"))
print("Naive:", naive, "| tzinfo:", naive.tzinfo)
print("Aware (Palmas):", palmas.isoformat())

# Same online class seen from other cities
for tz in ["UTC", "America/New_York", "Europe/London", "Asia/Tokyo"]:
    print(f"{tz:<20}", palmas.astimezone(ZoneInfo(tz)).strftime("%Y-%m-%d %H:%M %Z"))

Naive: 2026-10-22 19:30:00 | tzinfo: None
Aware (Palmas): 2026-10-22T19:30:00-03:00
UTC                  2026-10-22 22:30 UTC
America/New_York     2026-10-22 18:30 EDT
Europe/London        2026-10-22 23:30 BST
Asia/Tokyo           2026-10-23 07:30 JST


### 4.1 Horário de verão (DST) e a importância do UTC

Nova York e Londres mudam o relógio ao longo do ano; o Brasil não usa mais horário de verão desde 2019. Logo, a **diferença** entre Palmas e Nova York muda conforme a data. Armazenar tudo em **UTC** e converter só na exibição evita esse problema.

In [9]:
def utc_offset_hours(dt_local: datetime, tz_name: str) -> float:
    """Return the UTC offset (in hours) of a given instant in the target time zone."""
    return dt_local.astimezone(ZoneInfo(tz_name)).utcoffset().total_seconds() / 3600

for d in [datetime(2026, 1, 15, 12, tzinfo=timezone.utc), datetime(2026, 7, 15, 12, tzinfo=timezone.utc)]:
    print(d.date(), "| Palmas:", utc_offset_hours(d, "America/Araguaina"),
          "| New York:", utc_offset_hours(d, "America/New_York"),
          "| London:", utc_offset_hours(d, "Europe/London"))

now_utc = datetime.now(timezone.utc)
print("Current UTC instant (ISO):", now_utc.isoformat(timespec="seconds"))

2026-01-15 | Palmas: -3.0 | New York: -5.0 | London: 0.0
2026-07-15 | Palmas: -3.0 | New York: -4.0 | London: 1.0
Current UTC instant (ISO): 2026-10-22T00:34:29+00:00


## 5. Datas no pandas: `pd.to_datetime()`

O pandas usa o tipo `Timestamp` (baseado em `numpy.datetime64`, com precisão de nanossegundos). `pd.to_datetime()` converte strings, números ou colunas inteiras de uma vez — muito mais rápido que `strptime` em loop.

In [10]:
raw = pd.Series(["2026-02-09", "2026-03-14 22:47", "2026/04/01", "not a date", None])
parsed = pd.to_datetime(raw, errors="coerce", format="mixed")
print(parsed)
print("dtype:", parsed.dtype, "| invalid values (NaT):", parsed.isna().sum())

0   2026-02-09 00:00:00
1   2026-03-14 22:47:00
2   2026-04-01 00:00:00
3                   NaT
4                   NaT
dtype: datetime64[ns]
dtype: datetime64[ns] | invalid values (NaT): 2


> 📝 `errors="coerce"` transforma valores inválidos em `NaT` (*Not a Time*), o equivalente temporal do `NaN`. Para formatos brasileiros, use `dayfirst=True` ou, melhor ainda, um `format=` explícito.

In [11]:
br_series = pd.Series(["09/02/2026", "03/04/2026", "18/12/2026"])
print(pd.to_datetime(br_series, format="%d/%m/%Y"))

# Unix timestamps (seconds since 1970-01-01 UTC), common in app logs
epoch = pd.Series([1771804800, 1792800000])
print(pd.to_datetime(epoch, unit="s", utc=True))

0   2026-02-09
1   2026-04-03
2   2026-12-18
dtype: datetime64[ns]
0   2026-02-23 00:00:00+00:00
1   2026-10-24 00:00:00+00:00
dtype: datetime64[ns, UTC]


## 6. Gerando calendários reais: `pd.date_range` e `DatetimeIndex`

`pd.date_range()` gera sequências de datas seguindo regras de calendário reais: dias úteis (`B`), semanas ancoradas em segunda-feira (`W-MON`), fim de mês (`ME`), etc. É a base do nosso dataset.

In [12]:
print(pd.date_range("2026-02-09", periods=5, freq="D"))
print(pd.date_range("2026-02-09", "2026-02-20", freq="B"))           # business days
print(pd.date_range("2026-02-01", "2026-06-30", freq="ME"))           # month ends
print(pd.date_range("2026-02-09", periods=4, freq="W-MON"))           # Mondays

DatetimeIndex(['2026-02-09', '2026-02-10', '2026-02-11', '2026-02-12', '2026-02-13'], dtype='datetime64[ns]', freq='D')
DatetimeIndex(['2026-02-09', '2026-02-10', '2026-02-11', '2026-02-12', '2026-02-13', '2026-02-16', '2026-02-17',
               '2026-02-18', '2026-02-19', '2026-02-20'],
              dtype='datetime64[ns]', freq='B')
DatetimeIndex(['2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30'], dtype='datetime64[ns]', freq='ME')
DatetimeIndex(['2026-02-09', '2026-02-16', '2026-02-23', '2026-03-02'], dtype='datetime64[ns]', freq='W-MON')


### 6.1 Dias úteis com feriados nacionais

Com `pd.offsets.CustomBusinessDay` podemos excluir feriados. Abaixo, feriados nacionais brasileiros de 2026 que caem em dias de semana durante o ano letivo (Carnaval 16–17/02 é ponto facultativo, mas escolas geralmente não têm aula).

In [13]:
br_holidays_2026 = pd.to_datetime([
    "2026-02-16", "2026-02-17",  # Carnival
    "2026-04-03",                # Good Friday
    "2026-04-21",                # Tiradentes
    "2026-05-01",                # Labour Day
    "2026-06-04",                # Corpus Christi
    "2026-09-07",                # Independence Day
    "2026-10-12",                # Our Lady Aparecida / Children's Day
    "2026-11-02",                # All Souls' Day
    "2026-11-20",                # Black Consciousness Day
])
school_bday = pd.offsets.CustomBusinessDay(holidays=br_holidays_2026)
feb_school_days = pd.date_range("2026-02-09", "2026-02-28", freq=school_bday)
print("School days in Feb (after start):", len(feb_school_days))
print(feb_school_days.strftime("%d/%m (%a)").tolist())

School days in Feb (after start): 13
['09/02 (Mon)', '10/02 (Tue)', '11/02 (Wed)', '12/02 (Thu)', '13/02 (Fri)', '18/02 (Wed)', '19/02 (Thu)', '20/02 (Fri)', '23/02 (Mon)', '24/02 (Tue)', '25/02 (Wed)', '26/02 (Thu)', '27/02 (Fri)']


## 7. Construindo o dataset: progresso de estudantes de inglês em 2026

**Transparência:** este dataset é **sintético**. O calendário é real (ano letivo 2026, segundas-feiras, recesso de julho, feriados), mas os scores são gerados por uma regra explícita:

- Curva de aprendizagem logarítmica (ganhos rápidos no início, depois desaceleração);
- Cada estudante tem nível inicial (CEFR A1–B1) e ritmo próprio;
- No **recesso de julho** não há avaliações, e ocorre leve perda (*summer/winter learning loss*);
- Ruído aleatório com semente fixa (`seed=42`) para reprodutibilidade.

Os resultados ilustram técnicas; **não** representam alunos reais nem evidência educacional.

In [14]:
def build_school_weeks(year: int = 2026) -> pd.DatetimeIndex:
    """Return Monday assessment dates of a Brazilian school year, excluding the July break."""
    weeks = pd.date_range(f"{year}-02-09", f"{year}-12-14", freq="W-MON")
    july_break = (weeks >= f"{year}-07-06") & (weeks <= f"{year}-07-31")
    return weeks[~july_break]


def simulate_progress(weeks: pd.DatetimeIndex, seed: int = 42) -> pd.DataFrame:
    """Simulate weekly proficiency scores (0-100) with a log learning curve and a mid-year break loss."""
    rng = np.random.default_rng(seed)
    profiles = {  # student_id: (cefr_start, baseline, weekly_rate)
        "S01": ("A1", 22, 9.0), "S02": ("A1", 25, 7.0), "S03": ("A2", 38, 8.0),
        "S04": ("A2", 42, 6.0), "S05": ("B1", 55, 5.5), "S06": ("B1", 58, 4.0),
    }
    break_end = pd.Timestamp(f"{weeks[0].year}-07-31")
    rows = []
    for sid, (cefr, base, rate) in profiles.items():
        for i, wk in enumerate(weeks):
            score = base + rate * np.log1p(i)
            if wk > break_end:
                score -= 3.0  # learning loss after the break
            score += rng.normal(0, 2.5)
            rows.append({"student_id": sid, "cefr_start": cefr, "assessment_date": wk,
                         "score": float(np.clip(score, 0, 100))})
    return pd.DataFrame(rows)


weeks = build_school_weeks()
progress = simulate_progress(weeks)
print("Assessment weeks:", len(weeks), "| rows:", len(progress))
progress.head()

Assessment weeks: 41 | rows: 246


,student_id,cefr_start,assessment_date,score
0,S01,A1,2026-02-09,22.761793
1,S01,A1,2026-02-16,25.638364
2,S01,A1,2026-02-23,33.763639
3,S01,A1,2026-03-02,36.828061
4,S01,A1,2026-03-09,31.607353


### 7.1 Simulando datas como texto e convertendo

Na prática, exportações de plataformas costumam trazer datas como **texto** no formato brasileiro e horários de envio. Vamos simular isso e reconverter com `pd.to_datetime`.

In [15]:
rng = np.random.default_rng(7)
progress["submitted_at"] = (
    progress["assessment_date"]
    + pd.to_timedelta(rng.integers(0, 4, len(progress)), unit="D")
    + pd.to_timedelta(rng.integers(8 * 60, 23 * 60, len(progress)), unit="min")
).dt.strftime("%d/%m/%Y %H:%M")   # exported as Brazilian-format text

print(progress[["assessment_date", "submitted_at"]].dtypes)
progress["submitted_at"] = pd.to_datetime(progress["submitted_at"], format="%d/%m/%Y %H:%M")
print(progress[["assessment_date", "submitted_at"]].dtypes)
progress.sample(5, random_state=1)

assessment_date    datetime64[ns]
submitted_at               object
dtype: object
assessment_date    datetime64[ns]
submitted_at       datetime64[ns]
dtype: object


,student_id,cefr_start,assessment_date,score,submitted_at
67,S02,A1,2026-09-07,43.914979,2026-09-07 12:59:00
245,S06,B1,2026-12-14,66.635732,2026-12-17 13:42:00
208,S06,B1,2026-03-02,61.349797,2026-03-03 08:46:00
222,S06,B1,2026-06-08,70.002920,2026-06-11 10:04:00
90,S03,A2,2026-04-06,57.241259,2026-04-06 13:38:00


## 8. Extraindo componentes com o acessor `.dt`

O acessor `.dt` expõe atributos temporais de uma coluna datetime: `.year`, `.month`, `.day_of_week`, `.quarter`, `.isocalendar().week`, `.month_name()`, entre outros.

In [16]:
dt = progress["submitted_at"].dt
progress = progress.assign(
    year=dt.year,
    month=dt.month,
    month_name=dt.month_name(),
    quarter=dt.quarter,
    day_of_week=dt.day_of_week,     # 0 = Monday
    day_name=dt.day_name(),
    iso_week=dt.isocalendar().week.astype(int),
    hour=dt.hour,
)
progress[["submitted_at", "month_name", "quarter", "day_name", "iso_week", "hour"]].head(8)

,submitted_at,month_name,quarter,day_name,iso_week,hour
0,2026-02-12 22:16:00,February,1,Thursday,7,22
1,2026-02-18 15:41:00,February,1,Wednesday,8,15
2,2026-02-25 09:18:00,February,1,Wednesday,9,9
3,2026-03-05 18:53:00,March,1,Thursday,10,18
4,2026-03-11 11:27:00,March,1,Wednesday,11,11
5,2026-03-19 11:23:00,March,1,Thursday,12,11
6,2026-03-26 10:53:00,March,1,Thursday,13,10
7,2026-03-30 10:58:00,March,1,Monday,14,10


### 8.1 Em que dia e horário os estudantes entregam as atividades?

Componentes extraídos permitem perguntas pedagógicas: os alunos entregam no mesmo dia da atividade ou deixam para depois? Estudam à noite?

In [17]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday"]
print(progress["day_name"].value_counts().reindex(order))
progress["period_of_day"] = pd.cut(progress["hour"], bins=[0, 12, 18, 24], right=False,
                                   labels=["morning", "afternoon", "evening"])
print(progress["period_of_day"].value_counts())

day_name
Monday       55
Tuesday      57
Wednesday    65
Thursday     69
Name: count, dtype: int64
period_of_day
afternoon    92
evening      84
morning      70
Name: count, dtype: int64


## 9. Aritmética de datas no pandas: `Timedelta`

Subtrair colunas datetime gera uma coluna `timedelta64`. Daqui calculamos o **atraso de entrega** e a **semana do curso** de cada avaliação.

In [18]:
progress["delay"] = progress["submitted_at"] - progress["assessment_date"]
progress["delay_hours"] = progress["delay"].dt.total_seconds() / 3600
progress["course_day"] = (progress["assessment_date"] - pd.Timestamp("2026-02-09")).dt.days
progress["course_week"] = progress["course_day"] // 7 + 1

print(progress[["assessment_date", "submitted_at", "delay", "delay_hours", "course_week"]].head())
print("\nMean delay per student (hours):")
print(progress.groupby("student_id")["delay_hours"].mean().round(1))

  assessment_date        submitted_at           delay  delay_hours  course_week
0      2026-02-09 2026-02-12 22:16:00 3 days 22:16:00    94.266667            1
1      2026-02-16 2026-02-18 15:41:00 2 days 15:41:00    63.683333            2
2      2026-02-23 2026-02-25 09:18:00 2 days 09:18:00    57.300000            3
3      2026-03-02 2026-03-05 18:53:00 3 days 18:53:00    90.883333            4
4      2026-03-09 2026-03-11 11:27:00 2 days 11:27:00    59.450000            5

Mean delay per student (hours):
student_id
S01    59.2
S02    49.0
S03    53.5
S04    51.8
S05    50.0
S06    60.6
Name: delay_hours, dtype: float64


### 9.1 Offsets de calendário vs. `Timedelta`

`pd.Timedelta(days=30)` soma uma **duração fixa**; `pd.DateOffset(months=1)` respeita o **calendário** (meses têm 28–31 dias). `MonthEnd`, `BusinessDay` etc. "ancoram" em regras.

In [19]:
ts = pd.Timestamp("2026-01-31")
print("+ Timedelta(30 days):", ts + pd.Timedelta(days=30))
print("+ DateOffset(months=1):", ts + pd.DateOffset(months=1))
print("+ MonthEnd(1):", pd.Timestamp("2026-02-10") + pd.offsets.MonthEnd(1))
print("+ 5 school days (holiday-aware):", pd.Timestamp("2026-04-17") + 5 * school_bday)

+ Timedelta(30 days): 2026-03-02 00:00:00
+ DateOffset(months=1): 2026-02-28 00:00:00
+ MonthEnd(1): 2026-02-28 00:00:00
+ 5 school days (holiday-aware): 2026-04-27 00:00:00


## 10. `DatetimeIndex` e seleção por datas

Quando as datas viram índice, ganhamos **seleção por strings parciais** (`"2026-03"` seleciona março inteiro) e fatiamento por intervalos.

In [20]:
s01 = progress.loc[progress["student_id"] == "S01"].set_index("assessment_date").sort_index()
print(type(s01.index), "| freq inferred:", pd.infer_freq(s01.index[:10]))
print(s01.loc["2026-03", ["score"]])
print("\nFirst semester rows:", len(s01.loc["2026-02":"2026-06"]),
      "| second semester rows:", len(s01.loc["2026-08":"2026-12"]))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'> | freq inferred: W-MON
                     score
assessment_date           
2026-03-02       36.828061
2026-03-09       31.607353
2026-03-16       34.870386
2026-03-23       39.832792
2026-03-30       39.924367

First semester rows: 21 | second semester rows: 20


## 11. Resampling: mudando a granularidade temporal

`resample()` funciona como um `groupby` temporal: agrupa por janelas de calendário (`ME` = mês, `QE` = trimestre) e aplica uma agregação. Para dados com vários alunos, combinamos com `pd.Grouper`.

In [21]:
monthly_s01 = s01["score"].resample("ME").agg(["mean", "min", "max", "count"]).round(1)
monthly_s01

,mean,min,max,count
assessment_date,,,,
2026-02-28,27.4,22.8,33.8,3
2026-03-31,36.6,31.6,39.9,5
2026-04-30,43.6,40.6,46.3,4
2026-05-31,46.5,44.8,48.6,4
2026-06-30,48.5,45.6,50.7,5
2026-07-31,NaN,NaN,NaN,0
2026-08-31,47.4,45.1,50.3,5
2026-09-30,50.2,49.9,50.7,4
2026-10-31,50.6,48.7,55.3,4


> 📝 Repare que **julho** aparece com `count = 0` e `NaN`: o resample cria todos os períodos do calendário, mesmo sem dados. Isso deixa visível o recesso escolar, em vez de escondê-lo.

In [22]:
monthly_all = (
    progress.groupby(["student_id", pd.Grouper(key="assessment_date", freq="ME")])["score"]
    .mean().round(1)
    .unstack("student_id")
)
monthly_all.index = monthly_all.index.strftime("%Y-%m")
monthly_all

student_id,S01,S02,S03,S04,S05,S06
assessment_date,,,,,,
2026-02,27.4,29.3,41.2,47.7,59.0,58.6
2026-03,36.6,38.4,52.6,51.7,64.7,64.3
2026-04,43.6,41.1,56.8,54.6,66.2,67.8
2026-05,46.5,42.6,56.0,59.4,70.9,68.8
2026-06,48.5,45.4,61.6,59.9,72.5,69.6
2026-08,47.4,45.2,59.6,57.6,69.9,68.6
2026-09,50.2,44.8,61.7,57.4,72.1,68.5
2026-10,50.6,45.5,61.3,60.7,70.0,66.4
2026-11,51.5,47.6,64.2,59.5,72.7,67.1


### 11.1 Agrupando por trimestre e por `Period`

`to_period("Q")` converte datas em **períodos** (intervalos de tempo), ótimos para relatórios bimestrais/trimestrais.

In [23]:
progress["quarter_period"] = progress["assessment_date"].dt.to_period("Q")
quarterly = progress.pivot_table(index="cefr_start", columns="quarter_period",
                                 values="score", aggfunc="mean").round(1)
quarterly

quarter_period,2026Q1,2026Q2,2026Q3,2026Q4
cefr_start,,,,
A1,34.1,44.8,46.8,49.2
A2,49.3,58.2,59.0,61.9
B1,62.4,69.4,69.7,69.4


## 12. Janelas móveis (`rolling`) e variação percentual

Uma **média móvel** suaviza o ruído semanal e revela a tendência. Com `diff()` e `pct_change()` medimos o ganho entre avaliações.

In [24]:
progress = progress.sort_values(["student_id", "assessment_date"])
progress["rolling_4w"] = (progress.groupby("student_id")["score"]
                          .transform(lambda s: s.rolling(4, min_periods=1).mean()))
progress["weekly_gain"] = progress.groupby("student_id")["score"].diff()
progress[["student_id", "assessment_date", "score", "rolling_4w", "weekly_gain"]].head(8).round(2)

,student_id,assessment_date,score,rolling_4w,weekly_gain
0,S01,2026-02-09,22.76,22.76,NaN
1,S01,2026-02-16,25.64,24.20,2.88
2,S01,2026-02-23,33.76,27.39,8.13
3,S01,2026-03-02,36.83,29.75,3.06
4,S01,2026-03-09,31.61,31.96,-5.22
5,S01,2026-03-16,34.87,34.27,3.26
6,S01,2026-03-23,39.83,35.78,4.96
7,S01,2026-03-30,39.92,36.56,0.09


## 13. Aplicação prática: o recesso de julho afetou o progresso?

Comparamos, para cada estudante, a média das **4 últimas avaliações antes** do recesso com as **4 primeiras depois**. Lembrete: o efeito foi *programado* na simulação (−3 pontos), então aqui estamos verificando se as técnicas o **recuperam** — um bom teste de sanidade analítica.

In [25]:
def break_effect(df: pd.DataFrame, start: str = "2026-07-06", end: str = "2026-07-31", n: int = 4) -> pd.DataFrame:
    """Compare mean score of the last n assessments before and first n after a break, per student."""
    before = df[df["assessment_date"] < start].groupby("student_id").tail(n)
    after = df[df["assessment_date"] > end].groupby("student_id").head(n)
    out = pd.DataFrame({
        "before": before.groupby("student_id")["score"].mean(),
        "after": after.groupby("student_id")["score"].mean(),
    })
    out["change"] = out["after"] - out["before"]
    return out.round(2)

effect = break_effect(progress)
print("Mean change across students:", round(effect["change"].mean(), 2))
effect

Mean change across students: -1.19


,before,after,change
student_id,,,
S01,48.52,47.38,-1.14
S02,44.58,45.50,0.92
S03,61.54,59.40,-2.14
S04,58.29,56.70,-1.59
S05,72.17,68.81,-3.36
S06,69.51,69.66,0.15


> 🔍 **Interpretação honesta:** a perda programada foi de −3 pontos, mas a mudança média observada é menor e alguns alunos até sobem. Por quê? (1) a curva de aprendizagem continua crescendo entre as janelas comparadas; (2) o ruído (desvio-padrão 2,5) é da mesma ordem do efeito; (3) só temos 4 avaliações por janela. Lição: com poucos dados, um efeito real pode ficar **parcialmente mascarado** — e nenhuma conclusão causal deve ser tirada de comparações antes/depois simples.

### 13.1 Ganho total no ano e ganho por semana letiva

Usamos as datas para normalizar o ganho pelo **tempo efetivo de aula** (semanas com avaliação), não pelo tempo de calendário.

In [26]:
summary = progress.groupby(["student_id", "cefr_start"]).agg(
    first_date=("assessment_date", "min"),
    last_date=("assessment_date", "max"),
    first_score=("score", "first"),
    last_score=("score", "last"),
    n_assessments=("score", "size"),
).reset_index()
summary["calendar_days"] = (summary["last_date"] - summary["first_date"]).dt.days
summary["total_gain"] = summary["last_score"] - summary["first_score"]
summary["gain_per_assessed_week"] = summary["total_gain"] / (summary["n_assessments"] - 1)
summary.round(2)

,student_id,cefr_start,first_date,last_date,first_score,last_score,n_assessments,calendar_days,total_gain,gain_per_assessed_week
0,S01,A1,2026-02-09,2026-12-14,22.76,54.28,41,308,31.52,0.79
1,S02,A1,2026-02-09,2026-12-14,26.36,46.34,41,308,19.98,0.50
2,S03,A2,2026-02-09,2026-12-14,37.09,65.26,41,308,28.17,0.70
3,S04,A2,2026-02-09,2026-12-14,45.40,59.23,41,308,13.83,0.35
4,S05,B1,2026-02-09,2026-12-14,57.10,67.30,41,308,10.20,0.26
5,S06,B1,2026-02-09,2026-12-14,57.88,66.64,41,308,8.76,0.22


> 🧠 **Leitura linguística:** a curva logarítmica reflete uma ideia comum em aquisição de segunda língua: alunos iniciantes (A1) costumam ter ganhos mais visíveis no começo, enquanto em níveis intermediários (B1) o progresso mensurável desacelera — o famoso *intermediate plateau*. Aqui, isso é uma **premissa do modelo**, não uma descoberta.

## 14. Visualização de séries temporais

### 14.1 Evolução semanal com média móvel

O `matplotlib.dates` formata o eixo X com nomes de meses. A faixa sombreada destaca o recesso de julho.

In [27]:
fig, ax = plt.subplots(figsize=(11, 5))
palette = dict(zip(sorted(progress["student_id"].unique()), sns.color_palette("tab10", 6)))
for sid, grp in progress.groupby("student_id"):
    ax.plot(grp["assessment_date"], grp["score"], alpha=0.25, color=palette[sid])
    ax.plot(grp["assessment_date"], grp["rolling_4w"], lw=2, color=palette[sid],
            label=f"{sid} ({grp['cefr_start'].iat[0]})")
ax.axvspan(pd.Timestamp("2026-07-06"), pd.Timestamp("2026-07-31"), color="grey", alpha=0.2, label="July break")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set(title="Weekly English proficiency scores — 2026 school year (synthetic)",
       xlabel="Month", ylabel="Score (0–100)")
ax.legend(ncol=4, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

### 14.2 Média mensal por nível CEFR inicial (seaborn)

In [28]:
monthly_cefr = (progress.groupby(["cefr_start", pd.Grouper(key="assessment_date", freq="ME")])["score"]
                .mean().reset_index())
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(data=monthly_cefr, x="assessment_date", y="score", hue="cefr_start", marker="o", ax=ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set(title="Monthly mean score by starting CEFR level (synthetic)", xlabel="", ylabel="Mean score")
plt.tight_layout()
plt.show()

### 14.3 Heatmap: semana ISO × estudante

Um heatmap com semanas ISO no eixo X mostra o calendário inteiro de relance — inclusive as semanas sem avaliação (lacunas).

In [29]:
heat = progress.pivot_table(index="student_id", columns="iso_week", values="score")
heat = heat.reindex(columns=range(heat.columns.min(), heat.columns.max() + 1))  # expose missing weeks
fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(heat, cmap="viridis", ax=ax, cbar_kws={"label": "score"})
ax.set(title="Scores by ISO week (blank columns = July break)", xlabel="ISO week", ylabel="Student")
plt.tight_layout()
plt.show()

### 14.4 Padrão de entregas por dia da semana e horário

In [30]:
counts = (progress.groupby(["day_name", "period_of_day"], observed=False).size()
          .unstack().reindex(order))
counts.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Set2",
            title="Submissions by weekday and period of day (synthetic)")
plt.xlabel("")
plt.ylabel("Submissions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 15. Fusos horários no pandas: aulas online internacionais

Imagine que os horários de envio foram registrados no fuso de Palmas. Para comparar com um professor parceiro em Londres, usamos `tz_localize` (atribui o fuso a datas naive) e `tz_convert` (converte entre fusos).

In [31]:
sub = progress[["student_id", "submitted_at"]].head(6).copy()
sub["local_palmas"] = sub["submitted_at"].dt.tz_localize("America/Araguaina")
sub["utc"] = sub["local_palmas"].dt.tz_convert("UTC")
sub["london"] = sub["local_palmas"].dt.tz_convert("Europe/London")
sub[["student_id", "local_palmas", "utc", "london"]]

,student_id,local_palmas,utc,london
0,S01,2026-02-12 22:16:00-03:00,2026-02-13 01:16:00+00:00,2026-02-13 01:16:00+00:00
1,S01,2026-02-18 15:41:00-03:00,2026-02-18 18:41:00+00:00,2026-02-18 18:41:00+00:00
2,S01,2026-02-25 09:18:00-03:00,2026-02-25 12:18:00+00:00,2026-02-25 12:18:00+00:00
3,S01,2026-03-05 18:53:00-03:00,2026-03-05 21:53:00+00:00,2026-03-05 21:53:00+00:00
4,S01,2026-03-11 11:27:00-03:00,2026-03-11 14:27:00+00:00,2026-03-11 14:27:00+00:00
5,S01,2026-03-19 11:23:00-03:00,2026-03-19 14:23:00+00:00,2026-03-19 14:23:00+00:00


> ⚠️ `tz_localize` **não** muda o relógio — apenas declara em que fuso aquele horário foi registrado. `tz_convert` muda o relógio, preservando o mesmo instante. Comparar colunas naive com aware gera `TypeError`: bom sinal de que o pandas está protegendo você.

In [32]:
# Exporting with ISO 8601 + offset keeps the timezone information in plain text
print(sub["utc"].dt.strftime("%Y-%m-%dT%H:%M:%S%z").head(3).tolist())
print(sub["local_palmas"].iloc[0].isoformat())

['2026-02-13T01:16:00+0000', '2026-02-18T18:41:00+0000', '2026-02-25T12:18:00+0000']
2026-02-12T22:16:00-03:00


## 16. Boas práticas com datas

1. **Armazene em UTC**, converta para o fuso local só na apresentação.
2. **Use ISO 8601** (`2026-10-22T19:30:00-03:00`) em arquivos e APIs.
3. **Declare o formato** em `pd.to_datetime(format=...)` — é mais rápido e evita trocar dia/mês.
4. Use `errors="coerce"` e **audite os `NaT`** gerados.
5. Prefira `DateOffset` para regras de calendário (meses, dias úteis) e `Timedelta` para durações fixas.
6. Ao fazer `resample`, verifique períodos vazios — eles podem ser recessos, falhas de coleta ou dados faltantes.
7. Documente feriados e calendários usados: a análise depende deles.

In [33]:
def validate_dates(series: pd.Series, fmt: str) -> dict:
    """Parse a text date series with an explicit format and report basic quality metrics."""
    parsed = pd.to_datetime(series, format=fmt, errors="coerce")
    return {
        "n": len(series),
        "invalid": int(parsed.isna().sum()),
        "min": parsed.min(),
        "max": parsed.max(),
        "future_dates": int((parsed > pd.Timestamp("2026-12-31")).sum()),
    }

print(validate_dates(pd.Series(["09/02/2026", "31/02/2026", "18/12/2026", "05/01/2027"]), "%d/%m/%Y"))

{'n': 4, 'invalid': 1, 'min': Timestamp('2026-02-09 00:00:00'), 'max': Timestamp('2027-01-05 00:00:00'), 'future_dates': 1}


## 17. Exercícios de fixação

1. Quantos **dias letivos** (dias úteis sem feriados) existem entre 09/02/2026 e 18/12/2026, excluindo o recesso de julho?
2. Qual estudante teve o **maior ganho médio semanal** no 2º semestre?
3. Converta o horário da aula (19h30 em Palmas) para o horário de **Sydney** em 22/10/2026.

In [34]:
# 1. School days in the 2026 school year, excluding July break
all_days = pd.date_range("2026-02-09", "2026-12-18", freq=school_bday)
school_days = all_days[(all_days < "2026-07-06") | (all_days > "2026-07-31")]
print("1) School days:", len(school_days))

# 2. Highest mean weekly gain in the second semester
second = progress[progress["assessment_date"] >= "2026-08-03"]
print("2) Mean weekly gain (2nd semester):")
print(second.groupby("student_id")["weekly_gain"].mean().round(2).sort_values(ascending=False).head(3))

# 3. Class time in Sydney
cls = datetime(2026, 10, 22, 19, 30, tzinfo=ZoneInfo("America/Araguaina"))
print("3) Sydney:", cls.astimezone(ZoneInfo("Australia/Sydney")).strftime("%Y-%m-%d %H:%M %Z"))

1) School days: 195
2) Mean weekly gain (2nd semester):
student_id
S01    0.27
S03    0.19
S02    0.04
Name: weekly_gain, dtype: float64
3) Sydney: 2026-10-23 09:30 AEDT


## 18. Resumo do dia

| Conceito | Ferramenta |
|---|---|
| Tipos básicos | `date`, `time`, `datetime`, `timedelta` |
| Texto ↔ data | `strptime`, `strftime`, `isoformat`, `pd.to_datetime(format=...)` |
| Calendários reais | `pd.date_range`, `CustomBusinessDay`, `DateOffset` |
| Componentes | `.dt.year`, `.dt.month`, `.dt.day_of_week`, `.dt.quarter`, `.dt.isocalendar()` |
| Agregação temporal | `resample`, `pd.Grouper`, `to_period`, `rolling` |
| Fusos horários | `zoneinfo`, `tz_localize`, `tz_convert`, UTC |

### 🔭 Limitações e próximos passos
- O dataset é **sintético**: as conclusões sobre o recesso e o *plateau* refletem as premissas do gerador, não alunos reais.
- Próximo passo natural: aplicar estas técnicas a um dataset público real com datas (por exemplo, séries temporais na fase `06_nlp_timeseries`) e estudar decomposição de tendência/sazonalidade.